In [ ]:
# ===== 패키지 설치 (최초 1회만 실행) =====
!pip install python-dotenv
!pip install -U langchain langchain-openai langchain-teddynote

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

### FewShotChatMessagePromptTemplate

In [3]:
examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다...",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다...",
        "answer": """...""",
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량...",
        "answer": """문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서...""",
    },
    {
        "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다...",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, ...",
    },
]

In [5]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import (
    SemanticSimilarityExampleSelector,
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

chroma = Chroma("fewshot_chat", OpenAIEmbeddings())

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{instruction}:\n{input}"),
        ("ai", "{answer}"),
    ]
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, OpenAIEmbeddings(), chroma, k=1,
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
)

In [6]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": " 2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

example_selector.select_examples(question)

[{'input': '2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다...',
  'instruction': '당신은 회의록 작성 전문가 입니다...',
  'answer': '...'}]

In [8]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.",
        ),
        few_shot_prompt,
        ("human", "{instruction}\n{input}"),
    ]
)

In [10]:
from langchain_teddynote.messages import stream_response

chain = final_prompt | llm # 체인 생성

answer = chain.stream(question) # 실행 및 결과 출력
stream_response(answer)

### ABC 기술 회사 제품 개발 팀 회의록
- 날짜: 2023년 12월 26일
- 참석자: 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)

#### 회의 주요 내용:
1. 프로젝트의 현재 진행 상황 검토
2. 다가오는 마일스톤에 대한 계획 수립
3. 각 팀원의 작업 영역 업데이트
4. 다음 주까지의 목표 설정

#### 회의 내용 요약:
- 최현수: 현재 프로젝트 진행 상황을 총괄하고, 팀원들과 소통하는 역할 수행
- 황지연: 주요 개발자로써 개발 진행 상황과 어려움을 보고하며, 추가 지원이 필요한 부분을 공유
- 김태영: UI/UX 디자이너로써 사용자 경험과 디자인에 관한 업데이트 제공

#### 다음 회의 일정:
- 다음 회의는 2023년 1월 2일(월) 오전 10시 예정
- 다음 주까지의 목표: 모든 핵심 기능 완료 및 UI/UX 디자인 완성

이상입니다.